In [1]:
import numpy as np
import pandas as pd
import os

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import sys
import os

# Get the absolute path to the directory containing the current script
try:
    current_file_path = os.path.abspath(__file__)
    current_script_dir = os.path.dirname(current_file_path)
except NameError:
    current_script_dir = os.getcwd()

# Navigate up two levels to get to the 'GlobalLocal' directory
project_root = os.path.abspath(os.path.join(current_script_dir, '..', '..'))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

from types import SimpleNamespace as ns

from src.analysis.stats import segregation_scatter as scat
from src.analysis.stats import stability_flexibility_anatomy as sfa

In [2]:
seg_run = "/hpc/home/jz421/coganlab/jz421/GlobalLocal/dcc_scripts/stats/results/Stimulus_-1.0to1.5sec_decFactor_8_outliers_10_drop_and_nan_thresh_perc_5.0_70.0-150.0_Hz_padLength_1.5s_filterbank_hilbert_stat_func_ttest_zmax_20/segregation_results/window_0.0to1.5s_all_lpfc_proportion_cohens_d_fdr_bh"
df = pd.read_csv(os.path.join(seg_run, 'per_split.csv'))

In [6]:
x = sfa.map_reliability(df)


KeyboardInterrupt: 

In [ ]:
from scipy.stats import spearmanr, pearsonr
def cosine_similarity(u, v):
    return np.dot(u,v) / (np.linalg.norm(u) * np.linalg.norm(v))

fn_method_mapping = {'spearman': lambda u, v: spearmanr(u, v)[0],
                        'pearson': lambda u, v: pearsonr(u, v)[0],
                        'cosine': cosine_similarity}
fn = fn_method_mapping['spearman']    

d = df.dropna(subset=['xA', 'xB', 'yA', 'yB']).copy()
# keep only electrodes defined on EVERY split, so each split's vectors are
# over the same units and the averages are comparable
n_sp = df['split'].nunique()
d = d[d.groupby('electrode')['split'].transform('size') == n_sp]
unit = 'electrode'

if d.empty:
    raise ValueError("no unit survived the completeness / parcel filter")

rows, rows_p = [], []



In [ ]:
for _, g in d.groupby('split'):
    if len(g) < min_units:
        continue
    xa, xb = g['xA'].to_numpy(), g['xB'].to_numpy()
    ya, yb = g['yA'].to_numpy(), g['yB'].to_numpy()
    rows.append((0.5 * (fn(xa, yb) + fn(xb, ya)),
                    fn(xa, xb), fn(ya, yb)))
    # The attenuation correction below is Pearson algebra, so it is always
    # computed on Pearson correlations even when `method='spearman'` --
    # see `between_noise_corrected` in the docstring.
    rows_p.append((0.5 * (pearsonr(xa, yb)[0] + pearsonr(xb, ya)[0]),
                    pearsonr(xa, xb)[0], pearsonr(ya, yb)[0]))
if not rows:
    raise ValueError(f"every split has fewer than min_units={min_units} {unit}s")
between, rel_x, rel_y = np.nanmean(np.array(rows, float), axis=0)
arr = np.array(rows, float)
arr_p = np.array(rows_p, float)
between_p, rel_xp, rel_yp = np.nanmean(arr_p, axis=0)
# the values the attenuation ratio is built from: Pearson stands in for
# Spearman (see the docstring); the cosine is linear, so it is used as is.
arr_c = arr if method == 'cosine' else arr_p

def _corrected(b, rx, ry):
    return b / np.sqrt(rx * ry) if (rx > 0 and ry > 0) else np.nan

corrected = _corrected(*np.nanmean(arr_c, axis=0))

# The ratio is unstable when the reliabilities are small, so report a
# bootstrap interval over splits next to the point estimate rather than the
# point estimate alone.
if np.isfinite(corrected):
    rng = np.random.default_rng(0)
    boot = np.array([_corrected(*arr_c[rng.integers(0, len(arr_c), len(arr_c))].mean(axis=0))
                        for _ in range(2000)], float)
    ci = (tuple(float(v) for v in np.nanpercentile(boot, [2.5, 97.5]))
            if np.isfinite(boot).any() else (np.nan, np.nan))
else:
    # The point estimate is undefined, so a bootstrap of it is not an
    # interval for anything -- resamples that happen to land on a tiny
    # positive reliability produce arbitrarily large ratios.
    ci = (np.nan, np.nan)

note = None
if not np.isfinite(corrected):
    note = ("noise correction undefined: a split-half reliability is <= 0, so "
            "neither map is measured well enough to bound a correlation. "
            "Report `between` and the reliabilities, not a corrected value.")
elif corrected > 1.0:
    note = (f"noise-corrected value {corrected:.3f} exceeds 1, which is out of "
            "range for a correlation. At reliabilities this low the ratio is "
            "not estimable; report it as 'at the ceiling' and quote `between` "
            "and the reliabilities instead of the ratio.")

return dict(between=float(between), reliability_lwpc=float(rel_x),
            reliability_lwps=float(rel_y),
            between_noise_corrected=(float(corrected)
                                        if np.isfinite(corrected) else np.nan),
            between_noise_corrected_ci=ci,
            between_pearson=float(between_p),
            reliability_lwpc_pearson=float(rel_xp),
            reliability_lwps_pearson=float(rel_yp),
            note=note,
            unit=unit, n_units=int(d[unit].nunique()),
            n_splits=int(len(rows)), method=method)